# Bodo News Extractive Summarization — IndicBERT version
### Same goal, different engine: contextual embeddings instead of TF-IDF

**What changed vs. the TF-IDF version:**
- Steps 1–2 (cleaning, sentence tokenization) are **identical** — Bodo text still needs the same Devanagari + danda handling.
- Step 3 (stopwords) is now **optional** — a transformer already learns to down-weight function words from context, so it's not required for correctness, but you can still strip stopwords before embedding if you want cleaner input.
- Step 4 (TF-IDF Vectorizer) → **replaced** with IndicBERT sentence embeddings. There's no "fit on the corpus" step anymore — a pretrained model already understands Bodo, so we compute embeddings per-article, on demand.
- Step 5 (sum TF-IDF weights) → **replaced** with **cosine similarity to a document-centroid embedding**. The document's "meaning" is the average of all its sentence embeddings; the sentences most similar to that average are the ones that best represent the whole article.
- The 30%-minimum rule and reorder-into-original-order logic stay exactly the same.

**Model used:** `ai4bharat/IndicBERTv2-MLM-only` — trained on IndicCorp v2, covers 26 languages including Bodo (brx).

**Tip:** Go to `Runtime > Change runtime type > T4 GPU` before running — embedding lookups are much faster on GPU, especially if you batch-summarize the whole dataset.

In [27]:
# Install dependencies
!pip install -q transformers torch sentencepiece ipywidgets pandas numpy scikit-learn

In [28]:
import re
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import ipywidgets as widgets
from IPython.display import display, clear_output

pd.set_option('display.max_colwidth', 150)

## Step 0: Upload the dataset

Same as before — `Bodo_news.csv` with columns `title` and `article`.

In [29]:
from google.colab import files

uploaded = files.upload()   # choose Bodo_news.csv

df = pd.read_csv('Bodo_news.csv')
df = df.dropna(subset=['article']).reset_index(drop=True)
df['title'] = df['title'].fillna("")
print("Shape:", df.shape)
df.head()

Saving Bodo_news.csv to Bodo_news (1).csv
Shape: (2056, 5)


,title,article,source,url,sentence_count
0,सि ए ए जों दफा नं 6 खौ लानानै राज्यो-राजाखौ दाफोथायहो : समुज्जल भट्टाचार्य,राज्योआव दासान्दि मागु दमासि खुंफुंनायनि लाहार फाहार सोलिगासिनो दं बै दुलाराय आसाम फरायसा आफादनि दैदेनगिरि मोना सोरखारनि हेंथायै गोख्रों रावजों सा...,The Sentinel,https://bodo.sentinelassam.com/गाहाय-रादाब/dont-confuse-people-by-linking-clause-6-with-caa-samujjal-bhattacharjya,16
1,लखनौनि सा 11 राजखान्थि बादायगिरिया के बि सि आव फिन हरनायाव फेलें,बलीउदनि सावथुन मुलुगनि राफोद फावखुंगुर अमिताभ बश्शननि स्व' कौन बनेगा करोड़पति (KBC) हाबाफारिया जौसे जांख्लायाव गाखोहैदों। टि आर पि (Television Rat...,The Sentinel,https://bodo.sentinelassam.com/रंजानाय/11-politician-from-lucknow-failed-to-answer-in-kbc,21
2,बलीउदनि फावखुंगुर सालमान खाननि रेस 3 सावथुनाव सिद्धार्थ मालहत्रनि बदलै बबी देउलखौ लाबोबाय,"बलीउद सावथुन मुलुगनि राफोद फावखुंगुर सालमान खाननि रन्सायनायाव रेस 3 सावथुना गुबुन गुबुन बिथिंनिफ्राय गोबां दानख'थाइ फिन्जाथाय नुनो मोननाय जादों, न...",The Sentinel,https://bodo.sentinelassam.com/रंजानाय/bollywood-star-bobby-deol-replaced-sidharth-malhotra-in-race-3-movie-of-salman-khan,14
3,मन्थ्रि हिमन्त आ समुज्जल भट्टाचार्यखौ का & एन आर सि सायाव डिबेटाव बाहागो लानो सेलेन्ज होदों,दासान्दि बलीउदनि सावथुन मुलुगाव सोहैबाय नोगोरारि सोदांफिन्नाय आइननि दान्थे बिदान्थे गोनां आयदाया। फेस्ला फावगिरि (Comedian) अभिनिट मिस्र आ आसामनि ...,The Sentinel,https://bodo.sentinelassam.com/हरखाब-रादाब/himanta-biswa-sarma-challenges-samujjal-bhattacharjya-for-debate-over-caanrc,7
4,गुवाहाटी क्लाबाव सि ए ए नि हेंथायै आसुनि बान्जायनायाव सुबुं जाउरिखांनाय हाबाफारि,गुवाहाटी नोगोरमानि साहित्यरथी रसराज लक्ष्मीनाथ बेजबरुवा फोथाराव बृहसप्तिबार खालि रन'सिङा अदेबानि सुबुं जाउरिखां हाबाफारियाव गासैबो थाखोनि राज्यो-र...,The Sentinel,https://bodo.sentinelassam.com/गाहाय-रादाब/hundreds-participate-in-ranasinga-against-caa-at-guwahati-club,16


## Step 1: Text Cleaning & Preprocessing (unchanged)

Keep only Devanagari characters, digits, and danda (`।`) — strips English datelines and stray symbols.

In [30]:
def clean_text(text):
    """Keep only Devanagari characters, digits, danda (।), apostrophe (’ or '), and whitespace."""
    if not isinstance(text, str):
        return ""
    # Remove anything that is NOT Devanagari, a digit, danda, apostrophe (U+2019 or U+0027), or whitespace
    text = re.sub(r'[^\u0900-\u097F0-9।\s\u2019\u0027]', ' ', text)
    # Collapse multiple spaces/newlines into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Quick test
sample_raw = "GUWAHATI, May 15: बर'लोगोसे बर' आयदा गाहायनि थाखाय गोलाव सोलेगोन!! क’क्राझार ल’निडीन बर’नि बर’ बिफानाव"
print("BEFORE:", sample_raw)
print("AFTER :", clean_text(sample_raw))

BEFORE: GUWAHATI, May 15: बर'लोगोसे बर' आयदा गाहायनि थाखाय गोलाव सोलेगोन!! क’क्राझार ल’निडीन बर’नि बर’ बिफानाव
AFTER : 15 बर'लोगोसे बर' आयदा गाहायनि थाखाय गोलाव सोलेगोन क’क्राझार ल’निडीन बर’नि बर’ बिफानाव


In [31]:
df['clean_article'] = df['article'].apply(clean_text)
df[['title', 'clean_article']].head()

,title,clean_article
0,सि ए ए जों दफा नं 6 खौ लानानै राज्यो-राजाखौ दाफोथायहो : समुज्जल भट्टाचार्य,राज्योआव दासान्दि मागु दमासि खुंफुंनायनि लाहार फाहार सोलिगासिनो दं बै दुलाराय आसाम फरायसा आफादनि दैदेनगिरि मोना सोरखारनि हेंथायै गोख्रों रावजों सा...
1,लखनौनि सा 11 राजखान्थि बादायगिरिया के बि सि आव फिन हरनायाव फेलें,बलीउदनि सावथुन मुलुगनि राफोद फावखुंगुर अमिताभ बश्शननि स्व' कौन बनेगा करोड़पति हाबाफारिया जौसे जांख्लायाव गाखोहैदों। टि आर पि आवबो साबसिन फिथाय नुन...
2,बलीउदनि फावखुंगुर सालमान खाननि रेस 3 सावथुनाव सिद्धार्थ मालहत्रनि बदलै बबी देउलखौ लाबोबाय,बलीउद सावथुन मुलुगनि राफोद फावखुंगुर सालमान खाननि रन्सायनायाव रेस 3 सावथुना गुबुन गुबुन बिथिंनिफ्राय गोबां दानख'थाइ फिन्जाथाय नुनो मोननाय जादों ना...
3,मन्थ्रि हिमन्त आ समुज्जल भट्टाचार्यखौ का & एन आर सि सायाव डिबेटाव बाहागो लानो सेलेन्ज होदों,दासान्दि बलीउदनि सावथुन मुलुगाव सोहैबाय नोगोरारि सोदांफिन्नाय आइननि दान्थे बिदान्थे गोनां आयदाया। फेस्ला फावगिरि अभिनिट मिस्र आ आसामनि रांखान्थि ब...
4,गुवाहाटी क्लाबाव सि ए ए नि हेंथायै आसुनि बान्जायनायाव सुबुं जाउरिखांनाय हाबाफारि,गुवाहाटी नोगोरमानि साहित्यरथी रसराज लक्ष्मीनाथ बेजबरुवा फोथाराव बृहसप्तिबार खालि रन'सिङा अदेबानि सुबुं जाउरिखां हाबाफारियाव गासैबो थाखोनि राज्यो र...


## Step 2: Custom Bodo Sentence Tokenization (unchanged)

In [32]:
def split_sentences(text):
    """Split cleaned Bodo text into a list of sentences using danda (।)."""
    if not text:
        return []
    parts = text.split('।')
    return [s.strip() for s in parts if s.strip()]

test_sentences = split_sentences(df.loc[0, 'clean_article'])
print(f"Article 0 has {len(test_sentences)} sentences")
for s in test_sentences[:3]:
    print("-", s)

Article 0 has 16 sentences
- राज्योआव दासान्दि मागु दमासि खुंफुंनायनि लाहार फाहार सोलिगासिनो दं बै दुलाराय आसाम फरायसा आफादनि दैदेनगिरि मोना सोरखारनि हेंथायै गोख्रों रावजों सावरायनाय जानायखौ नुनो मोननाय जादों
- बिसोरो बुंदोंदि नोगोरारि सोदांफिन्नाय आइन जों आसाम गोरोबथा नि दफा नं 6 खौ सरजाबनो नाङा होनना हुमखि होदों
- लोगोसे गोलाव 34 बोसोर नाङै भुगिनांनाय जेंनाखौ राज्योनि थागिबि सुबुं माहारिखौ संबिजिदारि रैखाथि थामोन्थाखौ रोखा खालामनो थाखाय माखासे थांखिफोर गथायदों


## Step 3: Stop-Words (optional, kept for pre-cleaning only)

With IndicBERT, function words are naturally down-weighted by context, so this step is no longer required for the math to work. It's kept here in case you want to strip obvious noise words before embedding — feel free to skip applying it.

In [33]:
BODO_STOPWORDS = [
    "आरो", "एबा", "अब्ला", "ऐ", "दा", "नो", "नों", "आं", "बिसोरो", "मोन",
    "बे", "बेफोर", "जेरै", "सोर", "बबे", "बेबादि", "आव", "जों", "नि", "सिम",
    "से", "नै", "थाम", "ब्रै", "नाथाइ", "दं", "गोन", "माब्ला", "मानोना",
    "बै", "आंनि", "थाखाय", "हागोन", "थानाय"
]
print(f"{len(BODO_STOPWORDS)} stopwords loaded (optional use)")

34 stopwords loaded (optional use)


## Step 4: Sentence Embeddings via IndicBERT

We load the pretrained model once. For each sentence we:
1. Tokenize it.
2. Run it through IndicBERT to get per-token hidden states.
3. **Mean-pool** the token embeddings (weighted by the attention mask, so padding doesn't skew the result) into a single vector per sentence.
4. L2-normalize the vector, so a plain dot product later equals cosine similarity.

This is the standard way to get sentence-level vectors out of a base BERT-style model (which, unlike Sentence-BERT, has no built-in pooling head).

In [34]:
MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded. Hidden size:", model.config.hidden_size)

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Hidden size: 768


In [35]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

@torch.no_grad()
def get_embeddings(sentences, batch_size=16, max_length=128):
    """Return an (n_sentences, hidden_size) tensor of L2-normalized sentence embeddings."""
    all_embeds = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        ).to(device)
        out = model(**enc)
        embeds = mean_pooling(out.last_hidden_state, enc['attention_mask'])
        embeds = F.normalize(embeds, p=2, dim=1)
        all_embeds.append(embeds.cpu())
    return torch.cat(all_embeds, dim=0)

# Quick test
test_embeds = get_embeddings(test_sentences)
print("Embeddings shape:", test_embeds.shape)

Embeddings shape: torch.Size([16, 768])


## Step 5: Sentence Ranking & Summary Extraction (embedding version)

For each article:
1. Embed every sentence (Step 4).
2. Compute the **document embedding** as the mean of all sentence embeddings — this represents "what the article is about" as a single point in vector space.
3. Score each sentence by its **cosine similarity** to that document embedding — sentences closest to the article's overall meaning score highest (this replaces "sum of TF-IDF weights").
4. Select `ceil(ratio × total_sentences)` top-scoring sentences (same 30%-minimum rule as before, so any article length gets a meaningful summary).
5. Re-sort the selected sentences back into original order and join with `।`.

In [36]:
def summarize_article_bert(raw_text, ratio=0.3, min_sentences=1, batch_size=16):
    cleaned = clean_text(raw_text)
    sentences = split_sentences(cleaned)
    n = len(sentences)

    if n == 0:
        return ""
    if n <= 2:
        return '।'.join(sentences) + '।'

    num_select = max(min_sentences, math.ceil(ratio * n))
    num_select = min(num_select, n)

    sent_embeds = get_embeddings(sentences, batch_size=batch_size)     # (n, dim), normalized
    doc_embed = sent_embeds.mean(dim=0, keepdim=True)
    doc_embed = F.normalize(doc_embed, p=2, dim=1)

    scores = (sent_embeds @ doc_embed.T).squeeze(1).numpy()            # cosine similarity
    top_idx = np.argsort(scores)[::-1][:num_select]
    selected_idx = sorted(top_idx)                                    # restore original order

    summary_sentences = [sentences[i] for i in selected_idx]
    return '।'.join(summary_sentences) + '।'

### Test on a sample article from the dataset

In [37]:
sample_idx = 0
original = df.loc[sample_idx, 'article']
summary = summarize_article_bert(original, ratio=0.3)

print("TITLE:", df.loc[sample_idx, 'title'])
print("\nORIGINAL ARTICLE:\n", original)
print("\nSUMMARY (30%):\n", summary)
print(f"\nOriginal sentences: {len(split_sentences(clean_text(original)))} | Summary sentences: {len(split_sentences(summary))}")

TITLE: सि ए ए जों दफा नं 6 खौ लानानै राज्यो-राजाखौ दाफोथायहो : समुज्जल भट्टाचार्य

ORIGINAL ARTICLE:
 राज्योआव दासान्दि मागु दमासि खुंफुंनायनि लाहार फाहार सोलिगासिनो दं बै दुलाराय आसाम फरायसा आफादनि दैदेनगिरि मोना सोरखारनि हेंथायै गोख्रों रावजों सावरायनाय जानायखौ नुनो मोननाय जादों। बिसोरो बुंदोंदि नोगोरारि सोदांफिन्नाय आइन (Citizenship Amendment Act) जों आसाम गोरोबथा (Assam Accord) नि दफा नं 6 खौ सरजाबनो नाङा होनना हुमखि होदों। लोगोसे गोलाव 34 बोसोर नाङै भुगिनांनाय जेंनाखौ राज्योनि थागिबि सुबुं माहारिखौ संबिजिदारि रैखाथि थामोन्थाखौ रोखा खालामनो थाखाय माखासे थांखिफोर गथायदों। गुवाहाटी नोगोरमानि सहीद नियासाव खौरांगिरिजों जानाय मुगा मुगि बुब्लियाव रबिबार खालि दुलाराय आसाम फरायसा आफादनि बोसोनगिरि (AASU Chief Adviser) समुज्जल भट्टाचार्य बिथाङा बुङो दफा नं 6 आ गुबैयै 1951 आरो 1971 मायथाइनि गेजेराव आसामाव हाबफैनाय आयेन बेरेखा नोगोरारिखौ सिनायथिनानै बोखारनाय आरो थागिबि सुबुं माहारिनि गांनाय उखैनायखौ सुफुंनाय आरो संबिजिदारि रैखाथि थामोनथा जगायनाय। गुबुन फारसेथिं जाय नोगोरारि सोदांफिन्नाय आइन (C

### (Optional) Generate summaries for the whole dataset

**Heads up:** unlike TF-IDF (near-instant), this runs a forward pass through IndicBERT for every sentence in every article. With a GPU runtime this is fine for ~2,000 short news articles; on CPU it will be noticeably slower. If you just want to sanity-check quality, try it on `df.head(50)` first.

In [38]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets (e.g., 80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

# Generate summaries for the test set
test_df['generated_summary'] = test_df['article'].apply(lambda x: summarize_article_bert(x, ratio=0.3))

print("\nTest set with generated summaries (first 5 rows):")
display(test_df[['title', 'generated_summary']].head())

Training set shape: (1644, 6)
Test set shape: (412, 6)

Test set with generated summaries (first 5 rows):


,title,generated_summary
184,"असम थुनलाइ आफादा गंसे हादोर, मोनसे राव खान्थिखौ सोंखारिदों",थुनलाइ आफादनि आफादगिरि पर्मानन्द राजबंची जोंनि गिबि राव एबा बिमा रावानो सिगाङाव थागोन बेनि थाखाय गुबुन रावखौ जाबसिन्नाय आयदा लानानै सावरायज्लायनान...
1724,क’कराझार जिल्लायाव सानै NDFB(S) नि सोद्रोमा हमदों,क’कराझार डिसेम्बर 10 सिरियै खौरां मोन्नानै क’कराझार जिल्ला पुलिसआ पातगांव आव थानाय 210 बेटेलियन कब्राखौ लानानै रिपु होबथानाय हाग्रामानि नं 6 आरो 7...
251,गिबि मन्थ्रि सर्बानन्द सन'वाला दैबानायाव लोमलांनाय थिलियाव गोजौ प्लेटफर्म लुफुंनो बिथोन होदों,गुवाहाटी सेप्टेमबर आसामनि गिबि मन्थ्रि आरो माजुलि बियाबनि एम एल ए सर्बानन्द सन'वाला राज्यो सोरखारनि खाजोना आरो अल'खद सामलाय बिफानखौ दैबानानि समाव ...
1121,बर’ आरो कार्बिफोरा समानै मोन्थायखौ मोन्नांगौ: प्रमद बर’,बेबादियैनो बर’फोरा बिमुंखौ सोदांनो थाखाय दाबि खालामबोगासिनो दं भारत सरकाराबो बेबादियैनो नि खौ नोजोर होनानै शेडियुल ट्रायब बिमुंआव सोदेरनो खोथा होद...
1486,ए एस इ बि एस सि गेलेग्रा हान्जाया सुपार लिग बिमुं आवग्रिबाय,आसाम हादोरसा मोब्लिब बर्ड गेलेमुं क्लाब सुंद'यै ए एस इ बि एस सि या खेलायाव देरहालांनो गासै गरथाम पइन्टिनि गोनां जायो मिजिंथिनाय बादिनो टाइटेलखौ आव...


In [39]:
# Full dataset (uncomment to run — can take a while on CPU)
# df['generated_summary'] = df['article'].apply(lambda x: summarize_article_bert(x, ratio=0.3))
# df[['title', 'article', 'generated_summary']].to_csv('bodo_news_with_summaries_indicbert.csv', index=False)
# print("Saved: bodo_news_with_summaries_indicbert.csv")

## Step 6: Test on an UNSEEN article

Because IndicBERT doesn't need corpus fitting, this works exactly the same way on text that was never in your dataset — that's the point of using a pretrained model.

In [40]:
unseen_article = """पेस्ट योर बर' आर्टिकल हियार टेस्टिंग नि थाखाय।"""  # <-- replace with a real unseen Bodo article

print(summarize_article_bert(unseen_article, ratio=0.3))

पेस्ट योर बर' आर्टिकल हियार टेस्टिंग नि थाखाय।


## Step 7: Interactive UI

Same interface as the TF-IDF version, wired to the IndicBERT summarizer instead.

In [41]:
article_input = widgets.Textarea(
    value='',
    placeholder='Paste a Bodo news article here...',
    layout=widgets.Layout(width='100%', height='220px')
)

ratio_slider = widgets.FloatSlider(
    value=0.3, min=0.1, max=0.9, step=0.05,
    description='Summary %:',
    style={'description_width': 'initial'}
)

summarize_button = widgets.Button(
    description='Summarize',
    button_style='success',
    icon='compress'
)

output_box = widgets.Output()

def on_summarize_clicked(b):
    with output_box:
        clear_output()
        text = article_input.value.strip()
        if not text:
            print("⚠️  Please paste an article first.")
            return
        result = summarize_article_bert(text, ratio=ratio_slider.value)
        n_orig = len(split_sentences(clean_text(text)))
        n_summ = len(split_sentences(result))
        print(f"Original sentences: {n_orig}  |  Summary sentences: {n_summ}  ({ratio_slider.value:.0%} target)\n")
        print("SUMMARY:\n")
        print(result)

summarize_button.on_click(on_summarize_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Bodo News Extractive Summarizer (IndicBERT)</h3>"),
    article_input,
    ratio_slider,
    summarize_button,
    output_box
]))

## Notes: TF-IDF vs IndicBERT — what actually changed and why

- **No corpus fitting.** TF-IDF needed all 2,000 articles to compute IDF weights before it could rank anything. IndicBERT is pretrained, so `summarize_article_bert()` works standalone on a single article — this is closer to how you'd deploy it in production.
- **Ranking signal.** TF-IDF asked "which words are rare/important across my dataset, and which sentences contain the most of them?" IndicBERT asks "which sentences are semantically closest to what this specific article, as a whole, is about?" This tends to handle paraphrasing and word variation better, since it's not purely lexical (word-overlap) matching.
- **Cost.** TF-IDF scoring is essentially free. IndicBERT requires a forward pass per sentence — trivial for one article in the UI, but noticeable if you batch-summarize the full dataset on CPU. Use a GPU runtime for that.
- **Stopwords matter less.** They're kept in this notebook for optional pre-cleaning, but the ranking itself no longer depends on them the way TF-IDF's math did.
- **Quality check**: if summaries look worse than the TF-IDF version on some articles, try `ai4bharat/IndicBERTv2-MLM-Back-TLM` or `ai4bharat/IndicBERTv2-SS` (same repo, different training objectives) as drop-in replacements for `MODEL_NAME` — no other code changes needed.
- **You can combine both**: use IndicBERT similarity as the primary score, and add a small TF-IDF component (e.g. `final_score = 0.7*bert_score + 0.3*tfidf_score`) if you want to keep some keyword-based signal too.

In [43]:
import warnings
warnings.filterwarnings('ignore')

# Install rouge-score if not already installed
try:
    from rouge_score import rouge_scorer
except ImportError:
    !pip install -q rouge-score
    from rouge_score import rouge_scorer

# Initialize the ROUGE scorer
# We'll use Rouge-1, Rouge-2, and Rouge-L metrics with stemming.
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

# Iterate through the test_df to calculate ROUGE scores
# Note: ensure 'clean_article' and 'generated_summary' columns exist in test_df
for idx, row in test_df.iterrows():
    reference = row['clean_article']  # Using the cleaned original article from the test set as reference
    hypothesis = row['generated_summary']

    if not reference or not hypothesis:
        # Skip if either reference or hypothesis is empty to avoid errors
        # and ensure meaningful scores
        continue

    scores = scorer.score(reference, hypothesis)

    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

# Calculate and print the average ROUGE scores
print(f"Average ROUGE-1 F-measure on test set: {np.mean(rouge1_scores):.4f}")
print(f"Average ROUGE-2 F-measure on test set: {np.mean(rouge2_scores):.4f}")
print(f"Average ROUGE-L F-measure on test set: {np.mean(rougeL_scores):.4f}")

Average ROUGE-1 F-measure on test set: 0.4819
Average ROUGE-2 F-measure on test set: 0.2970
Average ROUGE-L F-measure on test set: 0.4819
